# Solar filament segmentation: GPU U-Net (soft labels, native resolution)

Deep-learning extension of the CPU baseline (local validation PQ 0.1796). A ResNet34 U-Net with a
full-resolution branch is trained on the training polygons only (no spines, chirality labels,
external annotations, or test labels). Post-processing is tuned on the calibration subset of
`configs/split.json`, the held-out validation partition is scored exactly once, and a candidate CSV
is written. **Nothing is submitted to Kaggle by this notebook.**

How to run:
1. *Add Input* -> the `filament-segmentation-2026` competition data.
2. *Session options* -> Accelerator: a GPU (T4 or P100). Turn Internet on if you want ImageNet-pretrained
   weights; without it the notebook trains from random initialisation and says so.
3. *Run All*. The first full cell is a 2-minute debug run; the second is the real run.

Outputs land in `/kaggle/working/run1/`: `metrics.json`, `candidate_submission.csv`, `best.pt`.

In [ ]:
import importlib.util, json, os, subprocess, sys
from pathlib import Path

if importlib.util.find_spec('pycocotools') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pycocotools==2.0.11'])

import torch
INPUT_ROOT = Path(os.environ.get('FILAMENT_INPUT', '/kaggle/input'))
WORK_ROOT = Path(os.environ.get('FILAMENT_WORK', '/kaggle/working'))
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Only the debug run will execute; enable a GPU accelerator for the full run.')

In [ ]:
"""Image-only contrast segmentation and exact COCO-mask evaluation.

Only the training segmentation polygons supervise calibration. No spines,
chirality labels, external annotations, or test labels are consumed.
"""
import json
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
from pycocotools import mask as coco
cv2.setNumThreads(2)


def load_annotations(data):
    a = json.loads((Path(data)/'train/MAGFiLO_1.0_Annotations_kaggle2026_train.json').read_text())
    records = defaultdict(list); annotations = defaultdict(list)
    for x in a['images']: records[x['file_name']].append(x)
    for x in a['annotations']: annotations[x['image_id']].append(x)
    return records, annotations


def ground_truth(records, annotations):
    result = []
    for record in records:
        masks = [coco.merge(coco.frPyObjects(a['segmentation'],record['height'],record['width']))
                 for a in annotations[record['id']]]
        result.append((record['id'], masks))
    return result


def contrast_maps(path):
    im = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if im is None: raise ValueError(f'Cannot read {path}')
    x = im.astype(np.float32)/255
    # Infer the bright solar disk from this image, not coordinates or filenames.
    _, disk = cv2.threshold(im,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(disk,8)
    if n < 2: raise ValueError(f'No solar disk: {path}')
    mask = (labels == (1+np.argmax(stats[1:,cv2.CC_STAT_AREA]))).astype(np.uint8)
    contours,_ = cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    mask[:] = 0; cv2.drawContours(mask,contours,-1,1,cv2.FILLED)
    mask = cv2.erode(mask,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(25,25)))
    smooth = cv2.GaussianBlur(x,(0,0),1.0)
    # Local brightness deficit at two scales; denominator stabilizes limb contrast.
    maps=[]
    for sigma in (12.,32.,64.):
        background = cv2.GaussianBlur(smooth,(0,0),sigma)
        maps.append(np.clip((background-smooth)/np.maximum(background,.08),0,1))
    return im, mask, maps


def predict_from_maps(mask, maps, config):
    contrast = maps[config['scale']]
    binary = ((contrast > config['threshold']) & (mask>0)).astype(np.uint8)
    radius = config['closing_radius']
    if radius:
        k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(radius*2+1,radius*2+1))
        binary = cv2.morphologyEx(binary,cv2.MORPH_CLOSE,k)
    binary &= mask
    count, labels, stats,_ = cv2.connectedComponentsWithStats(binary,8)
    accepted=[]
    for lab in range(1,count):
        if stats[lab,cv2.CC_STAT_AREA] < config['min_area']: continue
        # Bound maximum component size to reject large image artifacts.
        if stats[lab,cv2.CC_STAT_AREA] > config.get('max_area',100000): continue
        elongation = config.get('min_elongation', 1.)
        if elongation > 1:
            x,y,w,h,_ = stats[lab]
            yy,xx=np.nonzero(labels[y:y+h,x:x+w]==lab)
            if len(xx)<3: continue
            eigen=np.linalg.eigvalsh(np.cov(np.stack([xx,yy])))
            ratio=np.sqrt((eigen[1]+1)/(eigen[0]+1))
            if ratio < elongation: continue
        accepted.append(lab)
    return labels,accepted


def encode_instances(labels, accepted):
    return [coco.encode(np.asfortranarray(labels==lab,dtype=np.uint8)) for lab in accepted]


def score_masks(gt, pred):
    ng,npred=len(gt),len(pred)
    if ng and npred:
        iou=coco.iou(pred,gt,[0]*ng).T
    else: iou=np.zeros((ng,npred),dtype=float)
    hit=iou > .5
    tp=int(hit.sum()); fp=int((hit.sum(axis=0)==0).sum()); fn=int((hit.sum(axis=1)==0).sum())
    total=float(iou[hit].sum())
    nonzero=iou[iou>0]
    return {'tp':tp,'fp':fp,'fn':fn,'sum_iou':total,
            'pq':total/(tp+.5*fp+.5*fn) if tp+.5*fp+.5*fn else 0.,
            'pair_iou':nonzero.tolist(),'pair_dice':(2*nonzero/(1+nonzero)).tolist(),
            'gt_overlap_degrees':(iou>0).sum(axis=1).tolist(),
            'pred_overlap_degrees':(iou>0).sum(axis=0).tolist()}


def aggregate(rows):
    sums={k:sum(r[k] for r in rows) for k in ('tp','fp','fn','sum_iou')}
    denom=sums['tp']+.5*sums['fp']+.5*sums['fn']
    sums['pq']=sums['sum_iou']/denom if denom else 0.
    return sums


"""Deep-learning extension: a soft-label U-Net trained on annotator polygons.

The classical baseline in ``core.py`` thresholds a local-contrast map. This module
replaces that pixel classifier with a ResNet34 U-Net trained only on the competition's
training polygons (no spines, chirality labels, external data, or test labels) and keeps
everything else identical: the date-grouped split in ``configs/split.json``, the micro-PQ
metric from ``core.score_masks``, and the CSV format written by ``run.predict``.

Design choices
--------------
* Native 2048 px resolution. Filaments are thin, so downscaling would cost IoU.
* Soft targets. Each image's target is the mean of its annotators' masks. The metric
  scores a prediction against every annotator set separately, so the consensus is the
  best single prediction to aim for.
* Hysteresis threshold, minimum area and limb erosion are tuned on the calibration
  subset only. The validation partition is evaluated once, at the end.
"""
import csv
import hashlib
import json
import time
from collections import defaultdict
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pycocotools import mask as coco



# --------------------------------------------------------------------------- configuration
@dataclass
class TrainConfig:
    """Training and inference settings for the U-Net."""

    crop: int = 512                 # training crop size in native pixels
    batch_size: int = 8
    epochs: int = 40
    steps_per_epoch: int = 150
    lr: float = 5e-4
    weight_decay: float = 1e-4
    positive_crop_prob: float = 0.7  # share of crops centred on an annotated pixel
    pretrained: bool = True          # ImageNet weights (needs internet; falls back to random)
    amp: bool = True                 # mixed precision on CUDA
    tta: bool = True                 # average over flips at inference
    workers: int = 3
    eval_every: int = 5              # epochs between quick tuning-set PQ checks
    eval_images: int = 40            # tuning images used for those quick checks
    seed: int = 2026


@dataclass
class PostConfig:
    """Turns a probability map into filament instances."""

    low: float = 0.5        # pixels above this form candidate regions
    high: float = 0.5       # a region is kept only if its peak probability reaches this
    min_area: int = 300
    max_area: int = 150000
    disk_erode: int = 10    # pixels removed from the solar limb
    closing_radius: int = 0


def seed_everything(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# --------------------------------------------------------------------------- data
def image_stats(img: np.ndarray) -> tuple:
    """Mean and standard deviation of the non-black pixels (the solar disk)."""
    pixels = img[img > 8].astype(np.float32)
    if pixels.size < 100:
        pixels = img.astype(np.float32).ravel()
    return float(pixels.mean()), float(pixels.std() + 1e-3)


def normalize(img: np.ndarray, mean: float, std: float) -> np.ndarray:
    return (img.astype(np.float32) - mean) / std


def solar_disk(img: np.ndarray) -> np.ndarray:
    """Filled solar-disk mask (uint8 0/1) from Otsu thresholding, as in core.contrast_maps."""
    _, bright = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    count, labels, stats, _ = cv2.connectedComponentsWithStats(bright, 8)
    if count < 2:
        return np.ones(img.shape, np.uint8)
    mask = (labels == 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])).astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask[:] = 0
    cv2.drawContours(mask, contours, -1, 1, cv2.FILLED)
    return mask


def erode_disk(mask: np.ndarray, radius: int) -> np.ndarray:
    if radius <= 0:
        return mask
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * radius + 1, 2 * radius + 1))
    return cv2.erode(mask, kernel)


def load_case(data_dir, name: str, records: dict, annotations: dict) -> dict:
    """Read one training image with its soft consensus target and per-annotator masks."""
    img = cv2.imread(str(Path(data_dir) / "train/train_images" / name), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Cannot read training image {name}")
    gt = ground_truth(records[name], annotations)
    target = np.zeros(img.shape, np.float32)
    for _, masks in gt:
        if masks:
            target += coco.decode(coco.merge(masks))
    target /= max(1, len(gt))
    target_u8 = np.round(target * 255).astype(np.uint8)
    positives = np.flatnonzero(target_u8.ravel() >= 128)
    if positives.size > 4096:
        positives = np.random.default_rng(0).choice(positives, 4096, replace=False)
    mean, std = image_stats(img)
    return {"name": name, "image": img, "target": target_u8, "gt": gt, "disk": solar_disk(img),
            "mean": mean, "std": std, "positives": positives.astype(np.int64)}


def load_cases(data_dir, names, records, annotations, log=print) -> list:
    cases = []
    for index, name in enumerate(names, 1):
        cases.append(load_case(data_dir, name, records, annotations))
        if index % 50 == 0 or index == len(names):
            log(f"loaded {index}/{len(names)} images")
    return cases


def sample_crop(case: dict, size: int, rng: np.random.Generator, positive_prob: float):
    """Random crop, centred near an annotated pixel with probability ``positive_prob``."""
    img, target = case["image"], case["target"]
    height, width = img.shape
    if case["positives"].size and rng.random() < positive_prob:
        flat = int(case["positives"][rng.integers(case["positives"].size)])
        center_y, center_x = divmod(flat, width)
        y0 = center_y - int(rng.integers(size // 8, size - size // 8))
        x0 = center_x - int(rng.integers(size // 8, size - size // 8))
    else:
        y0 = int(rng.integers(0, max(1, height - size + 1)))
        x0 = int(rng.integers(0, max(1, width - size + 1)))
    y0 = int(np.clip(y0, 0, max(0, height - size)))
    x0 = int(np.clip(x0, 0, max(0, width - size)))
    patch = normalize(img[y0:y0 + size, x0:x0 + size], case["mean"], case["std"])
    soft = target[y0:y0 + size, x0:x0 + size].astype(np.float32) / 255.0
    if patch.shape != (size, size):
        pad = ((0, size - patch.shape[0]), (0, size - patch.shape[1]))
        patch = np.pad(patch, pad, constant_values=-case["mean"] / case["std"])
        soft = np.pad(soft, pad)
    return patch, soft


def augment(patch: np.ndarray, soft: np.ndarray, rng: np.random.Generator):
    """Dihedral flips plus brightness, contrast and noise jitter."""
    if rng.random() < 0.5:
        patch, soft = patch[:, ::-1], soft[:, ::-1]
    if rng.random() < 0.5:
        patch, soft = patch[::-1], soft[::-1]
    if rng.random() < 0.5:
        patch, soft = patch.T, soft.T
    patch = patch * rng.uniform(0.8, 1.25) + rng.uniform(-0.2, 0.2)
    patch = patch + rng.normal(0.0, rng.uniform(0.0, 0.05), patch.shape)
    return patch.astype(np.float32), soft


class CropDataset(torch.utils.data.Dataset):
    """Endless stream of random augmented crops; deterministic given (seed, epoch, index)."""

    def __init__(self, cases: list, cfg: TrainConfig, length: int):
        self.cases, self.cfg, self.length, self.epoch = cases, cfg, length, 0

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    def __len__(self) -> int:
        return self.length

    def __getitem__(self, index: int):
        rng = np.random.default_rng([self.cfg.seed, self.epoch, index])
        case = self.cases[int(rng.integers(len(self.cases)))]
        patch, soft = sample_crop(case, self.cfg.crop, rng, self.cfg.positive_crop_prob)
        patch, soft = augment(patch, soft, rng)
        return (torch.from_numpy(np.ascontiguousarray(patch))[None],
                torch.from_numpy(np.ascontiguousarray(soft, dtype=np.float32))[None])


# --------------------------------------------------------------------------- model
def conv_block(c_in: int, c_out: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
    )


class UpBlock(nn.Module):
    """Upsample, concatenate the encoder skip connection, and convolve twice."""

    def __init__(self, c_in: int, c_skip: int, c_out: int):
        super().__init__()
        self.conv = conv_block(c_in + c_skip, c_out)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


class FilamentUNet(nn.Module):
    """ResNet34 encoder, U-Net decoder, and a shallow branch at full input resolution.

    The ResNet stem halves the resolution immediately, which blurs thin barbs; the extra
    full-resolution branch gives the head direct access to the raw pixels.
    """

    def __init__(self, pretrained: bool = True):
        super().__init__()
        import torchvision

        weights = torchvision.models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        try:
            net = torchvision.models.resnet34(weights=weights)
        except Exception as error:  # offline sessions cannot download the weights
            print(f"Pretrained weights unavailable ({type(error).__name__}); using random init.")
            net = torchvision.models.resnet34(weights=None)
        conv1 = nn.Conv2d(1, 64, 7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            conv1.weight.copy_(net.conv1.weight.sum(dim=1, keepdim=True))
        self.stem = nn.Sequential(conv1, net.bn1, net.relu)
        self.pool = net.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = net.layer1, net.layer2, net.layer3, net.layer4
        self.up4 = UpBlock(512, 256, 256)
        self.up3 = UpBlock(256, 128, 128)
        self.up2 = UpBlock(128, 64, 64)
        self.up1 = UpBlock(64, 64, 64)
        self.full = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.Conv2d(16, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.ReLU(inplace=True),
        )
        self.head = nn.Sequential(
            nn.Conv2d(64 + 16, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1),
        )

    def forward(self, x):
        height, width = x.shape[-2:]
        pad_h, pad_w = (-height) % 32, (-width) % 32
        if pad_h or pad_w:
            x = F.pad(x, (0, pad_w, 0, pad_h), mode="reflect")
        full = self.full(x)
        stem = self.stem(x)
        e1 = self.layer1(self.pool(stem))
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)
        d = self.up4(e4, e3)
        d = self.up3(d, e2)
        d = self.up2(d, e1)
        d = self.up1(d, stem)
        d = F.interpolate(d, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return self.head(torch.cat([d, full], dim=1))[..., :height, :width]


def bce_dice_loss(logits: torch.Tensor, target: torch.Tensor, eps: float = 1.0) -> torch.Tensor:
    """Binary cross-entropy plus batch soft-Dice; both accept soft targets."""
    bce = F.binary_cross_entropy_with_logits(logits, target)
    prob = torch.sigmoid(logits)
    inter = (prob * target).sum()
    dice = 1.0 - (2.0 * inter + eps) / (prob.sum() + target.sum() + eps)
    return bce + dice


# --------------------------------------------------------------------------- inference
@torch.no_grad()
def predict_proba(model: nn.Module, img: np.ndarray, device: torch.device, cfg: TrainConfig,
                  tta=None) -> np.ndarray:
    """Probability map (float16, native resolution), optionally averaged over flips."""
    use_tta = cfg.tta if tta is None else tta
    use_amp = cfg.amp and device.type == "cuda"
    model.eval()
    mean, std = image_stats(img)
    x = torch.from_numpy(normalize(img, mean, std))[None, None].to(device)
    flips = [(), (3,), (2,), (2, 3)] if use_tta else [()]
    total = None
    for dims in flips:
        inp = torch.flip(x, dims) if dims else x
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            out = model(inp)
        out = torch.sigmoid(out.float())
        if dims:
            out = torch.flip(out, dims)
        total = out if total is None else total + out
    return (total / len(flips))[0, 0].cpu().numpy().astype(np.float16)


def segment(prob: np.ndarray, disk: np.ndarray, post: PostConfig):
    """Probability map -> (label image, accepted labels), the format encode_instances expects."""
    binary = ((prob > post.low) & (erode_disk(disk, post.disk_erode) > 0)).astype(np.uint8)
    if post.closing_radius:
        size = post.closing_radius * 2 + 1
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    count, labels, stats, _ = cv2.connectedComponentsWithStats(binary, 8)
    accepted = []
    for label in range(1, count):
        x, y, w, h, area = stats[label]
        if area < post.min_area or area > post.max_area:
            continue
        if post.high > post.low:
            inside = labels[y:y + h, x:x + w] == label
            if float(prob[y:y + h, x:x + w][inside].max()) < post.high:
                continue
        accepted.append(label)
    return labels, accepted


# --------------------------------------------------------------------------- evaluation
def count_matches(gt_masks: list, pred: list) -> dict:
    """tp/fp/fn/sum_iou with the same strict IoU > 0.5 rule as core.score_masks, but light."""
    n_gt, n_pred = len(gt_masks), len(pred)
    if not (n_gt and n_pred):
        return {"tp": 0, "fp": n_pred, "fn": n_gt, "sum_iou": 0.0}
    iou = coco.iou(pred, gt_masks, [0] * n_gt).T
    hit = iou > 0.5
    return {"tp": int(hit.sum()), "fp": int((hit.sum(axis=0) == 0).sum()),
            "fn": int((hit.sum(axis=1) == 0).sum()), "sum_iou": float(iou[hit].sum())}


def evaluate_probs(cases: list, probs: dict, post: PostConfig, full: bool = False) -> list:
    """One row per annotator record, exactly as run.evaluate builds them."""
    rows = []
    for case in cases:
        labels, accepted = segment(probs[case["name"]], case["disk"], post)
        pred = encode_instances(labels, accepted)
        for record_id, masks in case["gt"]:
            score = score_masks(masks, pred) if full else count_matches(masks, pred)
            rows.append({"file_name": case["name"], "image_record": record_id, **score})
    return rows


def bootstrap_ci(rows: list, samples: int = 2000, seed: int = 2026):
    """Capture-date cluster bootstrap of micro PQ, as in run.evaluate."""
    groups = defaultdict(list)
    for row in rows:
        groups[row["file_name"][:8]].append(row)
    clusters = [aggregate(group) for group in groups.values()]
    rng = np.random.default_rng(seed)
    pqs = [aggregate([clusters[i] for i in rng.integers(0, len(clusters), len(clusters))])["pq"]
           for _ in range(samples)]
    return np.percentile(pqs, [2.5, 97.5]).tolist(), len(groups)


def tune_post(cases: list, probs: dict, start: PostConfig, log=print) -> PostConfig:
    """Coordinate search over post-processing settings, using the tuning subset only."""
    best = start
    best_pq = aggregate(evaluate_probs(cases, probs, best))["pq"]
    log(f"post start pq={best_pq:.4f} {asdict(best)}")

    def trial(**changes):
        nonlocal best, best_pq
        candidate = replace(best, **changes)
        pq = aggregate(evaluate_probs(cases, probs, candidate))["pq"]
        log(f"post trial {changes} pq={pq:.4f}")
        if pq > best_pq:
            best, best_pq = candidate, pq

    for threshold in (0.3, 0.4, 0.5, 0.6):
        trial(low=threshold, high=threshold)
    for area in (50, 150, 300, 600):
        trial(min_area=area)
    for margin in (0.0, 0.15, 0.3):
        trial(high=min(0.95, best.low + margin))
    for radius in (0, 10, 20):
        trial(disk_erode=radius)
    log(f"post best pq={best_pq:.4f} {asdict(best)}")
    return best


# --------------------------------------------------------------------------- training
def train_model(model, train_cases, tune_cases, cfg: TrainConfig, device, out_dir, log=print) -> list:
    """Train with BCE+Dice, keep the checkpoint with the best quick tuning-set PQ."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    use_amp = cfg.amp and device.type == "cuda"
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=cfg.lr, total_steps=cfg.epochs * cfg.steps_per_epoch, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    dataset = CropDataset(train_cases, cfg, cfg.steps_per_epoch * cfg.batch_size)
    eval_cases = tune_cases[: cfg.eval_images]
    best_pq, history = -1.0, []
    for epoch in range(cfg.epochs):
        dataset.set_epoch(epoch)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=cfg.batch_size, num_workers=cfg.workers,
            pin_memory=device.type == "cuda", drop_last=True)
        model.train()
        start, losses = time.time(), []
        for patch, soft in loader:
            patch, soft = patch.to(device, non_blocking=True), soft.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(patch)
            loss = bce_dice_loss(logits.float(), soft)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            losses.append(float(loss.detach()))
        record = {"epoch": epoch + 1, "loss": float(np.mean(losses)), "seconds": time.time() - start}
        if (epoch + 1) % cfg.eval_every == 0 or epoch + 1 == cfg.epochs:
            probs = {c["name"]: predict_proba(model, c["image"], device, cfg, tta=False) for c in eval_cases}
            record["tune_pq"] = aggregate(evaluate_probs(eval_cases, probs, PostConfig()))["pq"]
            if record["tune_pq"] > best_pq:
                best_pq = record["tune_pq"]
                torch.save(model.state_dict(), out_dir / "best.pt")
        history.append(record)
        log(record)
    torch.save(model.state_dict(), out_dir / "last.pt")
    if not (out_dir / "best.pt").exists():
        torch.save(model.state_dict(), out_dir / "best.pt")
    return history


# --------------------------------------------------------------------------- submission files
def list_images(directory) -> list:
    return sorted(p for p in Path(directory).iterdir() if p.suffix.lower() in (".jpg", ".jpeg"))


def write_submission(model, test_dir, post: PostConfig, cfg: TrainConfig, device, out_csv, log=print,
                     limit=None) -> dict:
    """Write one CSV row per predicted filament: ``filament_id,segmentation_rle`` (counts only)."""
    paths = list_images(test_dir)[:limit]
    manifest, start = [], time.time()
    with Path(out_csv).open("w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["filament_id", "segmentation_rle"])
        for index, path in enumerate(paths, 1):
            tick = time.time()
            img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
            labels, accepted = segment(predict_proba(model, img, device, cfg), solar_disk(img), post)
            instances = encode_instances(labels, accepted)
            for number, rle in enumerate(instances, 1):
                writer.writerow([f"{path.stem}_{number}", rle["counts"].decode("ascii")])
            manifest.append({"file_name": path.name, "instances": len(instances), "seconds": time.time() - tick})
            if index % 20 == 0:
                log(f"predicted {index}/{len(paths)}")
    return {"seconds": time.time() - start, "images": manifest}


def verify_submission(csv_path, test_dir) -> dict:
    """Decode and re-encode every mask; check headers, unique ids, and image coverage."""
    paths = {p.stem: p for p in list_images(test_dir)}
    sizes, seen, ids, count = {}, set(), set(), 0
    with Path(csv_path).open() as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != ["filament_id", "segmentation_rle"]:
            raise ValueError(f"Unexpected CSV header: {reader.fieldnames}")
        for row in reader:
            fid = row["filament_id"]
            if fid in ids:
                raise ValueError(f"Duplicate filament_id {fid}")
            ids.add(fid)
            name, number = fid.rsplit("_", 1)
            if name not in paths or int(number) < 1:
                raise ValueError(f"Bad filament_id {fid}")
            if name not in sizes:
                sizes[name] = cv2.imread(str(paths[name]), cv2.IMREAD_GRAYSCALE).shape
            counts = row["segmentation_rle"]
            if '"' in counts or "'" in counts:
                raise ValueError(f"Quote character in RLE for {fid}")
            rle = {"size": list(sizes[name]), "counts": counts.encode("ascii")}
            mask = coco.decode(rle)
            if not mask.any():
                raise ValueError(f"Empty mask for {fid}")
            if coco.encode(np.asfortranarray(mask))["counts"] != rle["counts"]:
                raise ValueError(f"RLE is not canonical for {fid}")
            seen.add(name)
            count += 1
    digest = hashlib.sha256(Path(csv_path).read_bytes()).hexdigest()
    return {"status": "passed", "rows": count, "test_images": len(paths),
            "images_with_predictions": len(seen), "sha256": digest}


# --------------------------------------------------------------------------- experiment
def debug_overrides(cfg: TrainConfig) -> TrainConfig:
    """Tiny settings for a quick end-to-end check of the whole pipeline."""
    return replace(cfg, crop=256, batch_size=2, epochs=2, steps_per_epoch=4, workers=0,
                   eval_every=1, eval_images=2, tta=False, pretrained=False)


def run_experiment(data_dir, split: dict, out_dir, cfg=None, debug: bool = False,
                   baseline_pq: float = 0.1796, log=print) -> dict:
    """Train, tune on the calibration subset, evaluate once on validation, write a candidate CSV.

    Nothing is submitted to Kaggle. The candidate CSV and metrics are written to ``out_dir``.
    """
    cfg = debug_overrides(cfg or TrainConfig()) if debug else (cfg or TrainConfig())
    data_dir, out_dir = Path(data_dir), Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed_everything(cfg.seed)
    log(f"device={device} debug={debug}")

    calibration = set(split["calibration"])
    train_names = [n for n in split["training"] if n not in calibration]
    tune_names, val_names = list(split["calibration"]), list(split["validation"])
    test_dir = data_dir / "test/test_images"
    if debug:
        train_names, tune_names, val_names = train_names[:6], tune_names[:4], val_names[:4]
    log(f"train={len(train_names)} tune={len(tune_names)} validation={len(val_names)}")

    records, annotations = load_annotations(data_dir)
    train_cases = load_cases(data_dir, train_names, records, annotations, log)
    tune_cases = load_cases(data_dir, tune_names, records, annotations, log)

    model = FilamentUNet(cfg.pretrained).to(device)
    history = train_model(model, train_cases, tune_cases, cfg, device, out_dir, log)
    del train_cases
    model.load_state_dict(torch.load(out_dir / "best.pt", map_location=device))

    log("tuning post-processing on the calibration subset")
    tune_probs = {c["name"]: predict_proba(model, c["image"], device, cfg) for c in tune_cases}
    post = tune_post(tune_cases, tune_probs, PostConfig(), log)
    tune_pq = aggregate(evaluate_probs(tune_cases, tune_probs, post))["pq"]
    del tune_cases, tune_probs

    log("evaluating once on the held-out validation partition")
    val_cases = load_cases(data_dir, val_names, records, annotations, log)
    val_probs = {c["name"]: predict_proba(model, c["image"], device, cfg) for c in val_cases}
    rows = evaluate_probs(val_cases, val_probs, post, full=True)
    summary = aggregate(rows)
    interval, clusters = bootstrap_ci(rows)
    summary.update({"pq_bootstrap_95ci": interval, "images": len(val_cases),
                    "annotation_sets": len(rows), "date_clusters": clusters})
    log(f"validation PQ {summary['pq']:.4f} (baseline {baseline_pq:.4f}) 95% CI {interval}")

    log("writing the candidate submission (not uploaded anywhere)")
    csv_path = out_dir / "candidate_submission.csv"
    manifest = write_submission(model, test_dir, post, cfg, device, csv_path, log, limit=4 if debug else None)
    verification = verify_submission(csv_path, test_dir)
    metrics = {"validation": summary, "tuning_pq": tune_pq, "baseline_validation_pq": baseline_pq,
               "post": asdict(post), "train_config": asdict(cfg), "history": history,
               "verification": verification}
    (out_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    (out_dir / "validation_rows.json").write_text(json.dumps(rows))
    (out_dir / "inference_manifest.json").write_text(json.dumps(manifest, indent=2))
    log(f"done: {verification}")
    return metrics

In [ ]:
SPLIT = json.loads(r'''{"seed":2026,"grouping":"capture date, stratified by year","validation":["20110119082634Lh.jpeg","20110123174914Mh.jpeg","20110203082634Lh.jpeg","20110215083814Th.jpeg","20110312082634Lh.jpeg","20110325171014Mh.jpeg","20110406082634Lh.jpeg","20110511063154Uh.jpeg","20110526122534Ch.jpeg","20110615133454Bh.jpeg","20110616133454Bh.jpeg","20110810063134Lh.jpeg","20111018063134Lh.jpeg","20111019063154Uh.jpeg","20111029063154Uh.jpeg","20111215063134Lh.jpeg","20111222063154Uh.jpeg","20120217063154Uh.jpeg","20120217181154Bh.jpeg","20120227161654Bh.jpeg","20120318063154Uh.jpeg","20120331142954Bh.jpeg","20120421063154Uh.jpeg","20120509163414Mh.jpeg","20120523163014Mh.jpeg","20120610175114Mh.jpeg","20120715123434Ch.jpeg","20120723095014Th.jpeg","20120729122634Ch.jpeg","20120730135454Bh.jpeg","20120811121434Ch.jpeg","20120901114934Ch.jpeg","20120907141954Bh.jpeg","20121129063154Uh.jpeg","20130116172814Mh.jpeg","20130221063134Lh.jpeg","20130304170614Mh.jpeg","20130404063134Lh.jpeg","20130429063154Uh.jpeg","20130716162214Mh.jpeg","20130725122934Ch.jpeg","20130805135854Bh.jpeg","20130807073114Th.jpeg","20130808140054Bh.jpeg","20130913142354Bh.jpeg","20130925143254Bh.jpeg","20131020080114Th.jpeg","20131028190734Ch.jpeg","20131030185354Bh.jpeg","20131108045254Uh.jpeg","20131201192134Ch.jpeg","20131204192154Bh.jpeg","20131229112414Th.jpeg","20140116180214Mh.jpeg","20140127180134Ch.jpeg","20140128174554Bh.jpeg","20140205180154Bh.jpeg","20140409081614Th.jpeg","20140523195814Mh.jpeg","20140525070514Th.jpeg","20140604195854Bh.jpeg","20140607112354Uh.jpeg","20140607195854Bh.jpeg","20140615195814Mh.jpeg","20140615195854Bh.jpeg","20140817195834Ch.jpeg","20140821195834Ch.jpeg","20141007224934Lh.jpeg","20141010195834Ch.jpeg","20141025031154Uh.jpeg","20141214201834Ch.jpeg","20141216222634Lh.jpeg","20141225024354Uh.jpeg","20141225172214Mh.jpeg","20150112081954Uh.jpeg","20150113105034Ch.jpeg","20150113155654Bh.jpeg","20150125172714Mh.jpeg","20150222090014Th.jpeg","20150222112034Ch.jpeg","20150225171114Mh.jpeg","20150324145954Bh.jpeg","20150527082014Th.jpeg","20150527161214Mh.jpeg","20150610082014Th.jpeg","20150615133554Bh.jpeg","20150630083214Th.jpeg","20150706083014Th.jpeg","20150916055454Uh.jpeg","20151106171514Mh.jpeg","20151118172314Mh.jpeg","20151119133734Ch.jpeg","20151209201754Bh.jpeg","20160229151254Bh.jpeg","20160302133734Ch.jpeg","20160309172214Mh.jpeg","20160317232434Lh.jpeg","20160320133714Th.jpeg","20160326143654Bh.jpeg","20160327170714Mh.jpeg","20160518011654Uh.jpeg","20160724135554Bh.jpeg","20160804133734Ch.jpeg","20160920230134Lh.jpeg","20161012133734Ch.jpeg","20161023133814Th.jpeg","20161110151454Bh.jpeg","20161210133714Th.jpeg","20170217165250Bh.jpeg","20170323041730Lh.jpeg","20170417014150Uh.jpeg","20170417232530Lh.jpeg","20170418013450Uh.jpeg","20170427154550Bh.jpeg","20170619012150Uh.jpeg","20170722235230Lh.jpeg","20170805133710Th.jpeg","20170927143450Bh.jpeg","20171004143950Bh.jpeg","20171014133730Ch.jpeg","20171027223030Lh.jpeg","20171121222130Lh.jpeg","20180309000910Mh.jpeg","20180801232450Bh.jpeg","20180825205030Ch.jpeg","20180925000030Lh.jpeg","20181102104630Ch.jpeg","20190609123330Ch.jpeg","20190626025430Lh.jpeg","20190701123830Ch.jpeg","20200110073230Lh.jpeg","20200619123010Th.jpeg","20210103223630Lh.jpeg","20210201020330Lh.jpeg","20210616142650Bh.jpeg","20210817165030Mh.jpeg","20211231223430Lh.jpeg","20220617185352Mh.jpeg","20220803074752Th.jpeg"],"calibration":["20110109104734Ch.jpeg","20110220082634Lh.jpeg","20110221173814Mh.jpeg","20110301082654Uh.jpeg","20110313082654Uh.jpeg","20110324082614Th.jpeg","20110327143554Bh.jpeg","20110421164914Mh.jpeg","20110509121334Ch.jpeg","20110711071214Th.jpeg","20110727063234Lh.jpeg","20111116063134Lh.jpeg","20111223085314Th.jpeg","20120127085214Th.jpeg","20120205172414Mh.jpeg","20120213063134Lh.jpeg","20120223063154Uh.jpeg","20120223152054Bh.jpeg","20120225172914Mh.jpeg","20120414115434Ch.jpeg","20120414175914Mh.jpeg","20120816072814Th.jpeg","20120928063134Lh.jpeg","20121108155954Bh.jpeg","20121114172254Bh.jpeg","20121118092514Th.jpeg","20121218063534Lh.jpeg","20130214111534Ch.jpeg","20130219063134Lh.jpeg","20130303063134Lh.jpeg","20130303063154Uh.jpeg","20130324015554Uh.jpeg","20130326063134Lh.jpeg","20130517063134Lh.jpeg","20130609123334Ch.jpeg","20130609173414Mh.jpeg","20130616070414Th.jpeg","20130903060834Lh.jpeg","20130903141654Bh.jpeg","20130905073714Th.jpeg","20130926060834Lh.jpeg","20131019080114Th.jpeg","20131025185414Mh.jpeg","20131031185614Mh.jpeg","20140212024554Uh.jpeg","20140225163314Th.jpeg","20140312025454Uh.jpeg","20140312081114Th.jpeg","20140414020354Uh.jpeg","20140414195834Ch.jpeg","20140710083214Th.jpeg","20140725195814Mh.jpeg","20140802072114Th.jpeg","20140816072814Th.jpeg","20140818195814Mh.jpeg","20141012015654Uh.jpeg","20141012195854Bh.jpeg","20141013195834Ch.jpeg","20141013195854Bh.jpeg","20141016022554Uh.jpeg","20141016195834Ch.jpeg","20141026024454Uh.jpeg","20141105195814Mh.jpeg","20150216171714Mh.jpeg","20150325015554Uh.jpeg","20150427162314Mh.jpeg","20150627133754Bh.jpeg","20150719182314Mh.jpeg","20150731082034Lh.jpeg","20150819082014Th.jpeg","20151011144554Bh.jpeg","20151105133734Ch.jpeg","20151107133734Ch.jpeg","20151224133734Ch.jpeg","20151226133734Ch.jpeg","20160103133734Ch.jpeg","20160110133714Th.jpeg","20160310133734Ch.jpeg","20160430133714Th.jpeg","20160430133734Ch.jpeg","20160516012454Uh.jpeg","20160530234734Lh.jpeg","20160604163414Mh.jpeg","20160703133714Th.jpeg","20160726135354Bh.jpeg","20160808133714Th.jpeg","20161223133734Ch.jpeg","20170503013050Uh.jpeg","20170612135150Bh.jpeg","20170801164710Mh.jpeg","20170804140650Bh.jpeg","20170809164910Mh.jpeg","20170831170750Bh.jpeg","20171008144250Bh.jpeg","20171022170710Mh.jpeg","20171025133730Ch.jpeg","20171025223130Lh.jpeg","20180112130330Ch.jpeg","20180805232510Mh.jpeg","20180923112330Ch.jpeg","20180924112030Ch.jpeg","20190611032810Mh.jpeg","20190926165810Mh.jpeg","20200806053830Lh.jpeg","20210329015250Uh.jpeg","20210329114430Ch.jpeg","20210331231510Mh.jpeg","20210620163530Mh.jpeg","20210815140550Bh.jpeg","20220104043050Uh.jpeg","20220618185332Bh.jpeg"],"training":["20110109104734Ch.jpeg","20110114105034Ch.jpeg","20110128174814Mh.jpeg","20110205082634Lh.jpeg","20110211084114Th.jpeg","20110213093414Th.jpeg","20110220082634Lh.jpeg","20110221173814Mh.jpeg","20110301082654Uh.jpeg","20110306082634Lh.jpeg","20110313082654Uh.jpeg","20110317082654Uh.jpeg","20110322005814Mh.jpeg","20110324082614Th.jpeg","20110326170914Mh.jpeg","20110327143554Bh.jpeg","20110410141654Bh.jpeg","20110421164914Mh.jpeg","20110425063154Uh.jpeg","20110429135554Bh.jpeg","20110502063154Uh.jpeg","20110509121334Ch.jpeg","20110510063134Lh.jpeg","20110517164514Mh.jpeg","20110520063154Uh.jpeg","20110521063134Lh.jpeg","20110523063154Uh.jpeg","20110525063154Uh.jpeg","20110530122834Ch.jpeg","20110601063134Lh.jpeg","20110603163414Mh.jpeg","20110617063134Lh.jpeg","20110619063134Lh.jpeg","20110621063134Lh.jpeg","20110623063134Lh.jpeg","20110625163814Mh.jpeg","20110707074514Th.jpeg","20110711071214Th.jpeg","20110724164614Mh.jpeg","20110727063234Lh.jpeg","20110728084714Th.jpeg","20110803164914Mh.jpeg","20110815140454Bh.jpeg","20110823120334Ch.jpeg","20110824141054Bh.jpeg","20110826074214Th.jpeg","20110902073614Th.jpeg","20110914063134Lh.jpeg","20110918112834Ch.jpeg","20110923112234Ch.jpeg","20110923165814Mh.jpeg","20111009110234Ch.jpeg","20111011112054Uh.jpeg","20111025080614Th.jpeg","20111027145954Bh.jpeg","20111107063154Uh.jpeg","20111112193254Bh.jpeg","20111114082314Th.jpeg","20111115063134Lh.jpeg","20111116063134Lh.jpeg","20111201063154Uh.jpeg","20111206102934Ch.jpeg","20111208063134Lh.jpeg","20111214063134Lh.jpeg","20111214132434Ch.jpeg","20111218085014Th.jpeg","20111223085314Th.jpeg","20111226155454Bh.jpeg","20111229063154Uh.jpeg","20120103104134Ch.jpeg","20120104063134Lh.jpeg","20120110155754Bh.jpeg","20120113104834Ch.jpeg","20120114063154Uh.jpeg","20120116063254Uh.jpeg","20120127085214Th.jpeg","20120131063154Uh.jpeg","20120205172414Mh.jpeg","20120206063134Lh.jpeg","20120213063134Lh.jpeg","20120215153654Bh.jpeg","20120218173614Mh.jpeg","20120223063154Uh.jpeg","20120223152054Bh.jpeg","20120225172914Mh.jpeg","20120306063154Uh.jpeg","20120313063154Uh.jpeg","20120323144054Bh.jpeg","20120328164614Mh.jpeg","20120330063134Lh.jpeg","20120330063154Uh.jpeg","20120403083114Th.jpeg","20120403164114Mh.jpeg","20120404063154Uh.jpeg","20120414115434Ch.jpeg","20120414175914Mh.jpeg","20120424072414Th.jpeg","20120424120134Ch.jpeg","20120427063154Uh.jpeg","20120427162614Mh.jpeg","20120430063154Uh.jpeg","20120519070814Th.jpeg","20120603123034Ch.jpeg","20120627162114Mh.jpeg","20120703070814Th.jpeg","20120712123634Ch.jpeg","20120720085114Th.jpeg","20120725122934Ch.jpeg","20120726071914Th.jpeg","20120802090314Th.jpeg","20120803102114Th.jpeg","20120803164614Mh.jpeg","20120810121634Ch.jpeg","20120813164714Mh.jpeg","20120814140454Bh.jpeg","20120814163114Mh.jpeg","20120816072814Th.jpeg","20120827085114Th.jpeg","20120829063134Lh.jpeg","20120910074014Th.jpeg","20120913165314Mh.jpeg","20120914063134Lh.jpeg","20120917142654Bh.jpeg","20120928063134Lh.jpeg","20121006063534Lh.jpeg","20121007131134Ch.jpeg","20121009165914Mh.jpeg","20121018063134Lh.jpeg","20121021145454Bh.jpeg","20121022145554Bh.jpeg","20121024164714Mh.jpeg","20121026063134Lh.jpeg","20121027063134Lh.jpeg","20121027165114Mh.jpeg","20121030150354Bh.jpeg","20121108155954Bh.jpeg","20121109103634Ch.jpeg","20121113103634Ch.jpeg","20121114172254Bh.jpeg","20121118092514Th.jpeg","20121207103034Ch.jpeg","20121208171514Mh.jpeg","20121218063534Lh.jpeg","20121219063154Uh.jpeg","20130107174854Bh.jpeg","20130109130334Ch.jpeg","20130110104734Ch.jpeg","20130111104734Ch.jpeg","20130201084914Th.jpeg","20130208063154Uh.jpeg","20130211063134Lh.jpeg","20130212153254Bh.jpeg","20130214111534Ch.jpeg","20130218220314Mh.jpeg","20130219063134Lh.jpeg","20130222082914Th.jpeg","20130225171214Mh.jpeg","20130303063134Lh.jpeg","20130303063154Uh.jpeg","20130305112734Ch.jpeg","20130306172214Mh.jpeg","20130314165714Mh.jpeg","20130319144654Bh.jpeg","20130324015554Uh.jpeg","20130326063134Lh.jpeg","20130327175114Mh.jpeg","20130405114734Ch.jpeg","20130407063154Uh.jpeg","20130411115134Ch.jpeg","20130419070254Uh.jpeg","20130423072414Th.jpeg","20130425135954Bh.jpeg","20130502063134Lh.jpeg","20130510134654Bh.jpeg","20130517063134Lh.jpeg","20130601063154Uh.jpeg","20130609123334Ch.jpeg","20130609173414Mh.jpeg","20130611063154Uh.jpeg","20130615002434Lh.jpeg","20130616070414Th.jpeg","20130628063134Lh.jpeg","20130630070714Th.jpeg","20130705161714Mh.jpeg","20130707134154Bh.jpeg","20130717123434Ch.jpeg","20130719140154Bh.jpeg","20130809072814Th.jpeg","20130810121534Ch.jpeg","20130811140254Bh.jpeg","20130813060834Lh.jpeg","20130824060834Lh.jpeg","20130828141354Bh.jpeg","20130901073614Th.jpeg","20130903060834Lh.jpeg","20130903141654Bh.jpeg","20130905073714Th.jpeg","20130919060834Lh.jpeg","20130921060834Lh.jpeg","20130923060834Lh.jpeg","20130926060834Lh.jpeg","20131006075314Th.jpeg","20131007075314Th.jpeg","20131011195414Mh.jpeg","20131013185334Ch.jpeg","20131016185334Ch.jpeg","20131017185334Ch.jpeg","20131019080114Th.jpeg","20131025185414Mh.jpeg","20131027025654Uh.jpeg","20131031185614Mh.jpeg","20131101222734Lh.jpeg","20131107043054Uh.jpeg","20131109190514Mh.jpeg","20131109222434Lh.jpeg","20131113004514Mh.jpeg","20131113185414Mh.jpeg","20131115185414Mh.jpeg","20131116185334Ch.jpeg","20131125192154Bh.jpeg","20131128192214Mh.jpeg","20131130192134Ch.jpeg","20131206084114Th.jpeg","20131207084214Th.jpeg","20131224192154Bh.jpeg","20131225065154Uh.jpeg","20140111224134Lh.jpeg","20140112180134Ch.jpeg","20140122180214Mh.jpeg","20140210180154Bh.jpeg","20140212024554Uh.jpeg","20140213024554Uh.jpeg","20140216230134Lh.jpeg","20140218230234Lh.jpeg","20140225163314Th.jpeg","20140312025454Uh.jpeg","20140312081114Th.jpeg","20140321195814Mh.jpeg","20140323020654Uh.jpeg","20140401014754Uh.jpeg","20140406195854Bh.jpeg","20140411013854Uh.jpeg","20140413195914Mh.jpeg","20140414020354Uh.jpeg","20140414195834Ch.jpeg","20140420085614Th.jpeg","20140428233234Lh.jpeg","20140430071914Th.jpeg","20140501195814Mh.jpeg","20140508195834Ch.jpeg","20140509071314Th.jpeg","20140509195814Mh.jpeg","20140513012554Uh.jpeg","20140522070614Th.jpeg","20140531195834Ch.jpeg","20140601195814Mh.jpeg","20140603070414Th.jpeg","20140605195854Bh.jpeg","20140609195854Bh.jpeg","20140701195814Mh.jpeg","20140705035754Uh.jpeg","20140709195814Mh.jpeg","20140710083214Th.jpeg","20140712195854Bh.jpeg","20140716195854Bh.jpeg","20140719071414Th.jpeg","20140720195834Ch.jpeg","20140723071614Th.jpeg","20140725195814Mh.jpeg","20140726071814Th.jpeg","20140802072114Th.jpeg","20140803234534Lh.jpeg","20140806195834Ch.jpeg","20140807195854Bh.jpeg","20140813064254Uh.jpeg","20140814195854Bh.jpeg","20140816072814Th.jpeg","20140818195814Mh.jpeg","20140903073614Th.jpeg","20140904195814Mh.jpeg","20140908175354Bh.jpeg","20140908231334Lh.jpeg","20140918195854Bh.jpeg","20140921195854Bh.jpeg","20140922195854Bh.jpeg","20140927014554Uh.jpeg","20141001195834Ch.jpeg","20141004075414Th.jpeg","20141006224634Lh.jpeg","20141009023454Uh.jpeg","20141012015654Uh.jpeg","20141012195854Bh.jpeg","20141013195834Ch.jpeg","20141013195854Bh.jpeg","20141016022554Uh.jpeg","20141016195834Ch.jpeg","20141022223334Lh.jpeg","20141024195834Ch.jpeg","20141026024454Uh.jpeg","20141027025054Uh.jpeg","20141027224834Lh.jpeg","20141105195814Mh.jpeg","20141106195854Bh.jpeg","20141112195854Bh.jpeg","20141116222234Lh.jpeg","20141129222134Lh.jpeg","20141210201814Mh.jpeg","20150116082034Lh.jpeg","20150119082034Lh.jpeg","20150130084914Th.jpeg","20150203084714Th.jpeg","20150207084414Th.jpeg","20150216171714Mh.jpeg","20150221155154Bh.jpeg","20150228082414Th.jpeg","20150302082034Lh.jpeg","20150305081954Uh.jpeg","20150308081954Uh.jpeg","20150314081954Uh.jpeg","20150315082034Lh.jpeg","20150323172814Mh.jpeg","20150325015554Uh.jpeg","20150327145114Th.jpeg","20150330081954Uh.jpeg","20150409081954Uh.jpeg","20150417082314Th.jpeg","20150421140454Bh.jpeg","20150423120034Ch.jpeg","20150427162314Mh.jpeg","20150501120634Ch.jpeg","20150518145654Bh.jpeg","20150531082034Lh.jpeg","20150614162514Mh.jpeg","20150617161314Mh.jpeg","20150620082014Th.jpeg","20150621161314Mh.jpeg","20150625082014Th.jpeg","20150625082034Lh.jpeg","20150626161514Mh.jpeg","20150627133754Bh.jpeg","20150629161614Mh.jpeg","20150716162114Mh.jpeg","20150718082014Th.jpeg","20150719182314Mh.jpeg","20150720082014Th.jpeg","20150727082014Th.jpeg","20150731082034Lh.jpeg","20150806082034Lh.jpeg","20150819082014Th.jpeg","20150820152254Bh.jpeg","20150825082034Lh.jpeg","20150830084134Lh.jpeg","20150913082014Th.jpeg","20150919165814Mh.jpeg","20150926111934Ch.jpeg","20150927111834Ch.jpeg","20151011144554Bh.jpeg","20151013144654Bh.jpeg","20151019223534Lh.jpeg","20151105133734Ch.jpeg","20151107133734Ch.jpeg","20151116201114Mh.jpeg","20151116224034Lh.jpeg","20151117172314Mh.jpeg","20151122172614Mh.jpeg","20151128133714Th.jpeg","20151128133734Ch.jpeg","20151128222134Lh.jpeg","20151212171154Bh.jpeg","20151213023454Uh.jpeg","20151219133734Ch.jpeg","20151224133734Ch.jpeg","20151226133734Ch.jpeg","20151228024254Uh.jpeg","20160101133714Th.jpeg","20160103133734Ch.jpeg","20160107133714Th.jpeg","20160110133714Th.jpeg","20160125154854Bh.jpeg","20160127225034Lh.jpeg","20160129225334Lh.jpeg","20160201025254Uh.jpeg","20160204133734Ch.jpeg","20160213153054Bh.jpeg","20160215133734Ch.jpeg","20160215175514Mh.jpeg","20160223173514Mh.jpeg","20160225173314Mh.jpeg","20160226173214Mh.jpeg","20160310133734Ch.jpeg","20160315133714Th.jpeg","20160401133734Ch.jpeg","20160401170314Mh.jpeg","20160402025254Uh.jpeg","20160403170114Mh.jpeg","20160408232234Lh.jpeg","20160411134234Ch.jpeg","20160430133714Th.jpeg","20160430133734Ch.jpeg","20160501143154Bh.jpeg","20160510163914Mh.jpeg","20160516012454Uh.jpeg","20160526011254Uh.jpeg","20160530234734Lh.jpeg","20160604163414Mh.jpeg","20160606163514Mh.jpeg","20160609133654Bh.jpeg","20160617133654Bh.jpeg","20160619133714Th.jpeg","20160627133734Ch.jpeg","20160628152654Bh.jpeg","20160629133714Th.jpeg","20160703133714Th.jpeg","20160714235434Lh.jpeg","20160720133734Ch.jpeg","20160726135354Bh.jpeg","20160808133714Th.jpeg","20160809164914Mh.jpeg","20160810234034Lh.jpeg","20160831133714Th.jpeg","20160925165814Mh.jpeg","20160925225634Lh.jpeg","20160927133734Ch.jpeg","20160928143554Bh.jpeg","20161002225034Lh.jpeg","20161013133734Ch.jpeg","20161105222634Lh.jpeg","20161108021454Uh.jpeg","20161109171714Mh.jpeg","20161116133714Th.jpeg","20161129022454Uh.jpeg","20161201160954Bh.jpeg","20161205133714Th.jpeg","20161217133734Ch.jpeg","20161222133714Th.jpeg","20161223133734Ch.jpeg","20161224024154Uh.jpeg","20161229133734Ch.jpeg","20161231133734Ch.jpeg","20170102024454Uh.jpeg","20170103030554Uh.jpeg","20170106133714Th.jpeg","20170107133714Th.jpeg","20170117175014Mh.jpeg","20170119031254Uh.jpeg","20170122034654Uh.jpeg","20170125133734Ch.jpeg","20170218173710Mh.jpeg","20170222133710Th.jpeg","20170310133730Ch.jpeg","20170310173450Bh.jpeg","20170316133730Ch.jpeg","20170322015550Uh.jpeg","20170327143550Bh.jpeg","20170329133730Ch.jpeg","20170408133710Th.jpeg","20170411013550Uh.jpeg","20170415145450Bh.jpeg","20170423014350Uh.jpeg","20170425133730Ch.jpeg","20170502233130Lh.jpeg","20170503013050Uh.jpeg","20170609133710Th.jpeg","20170612135150Bh.jpeg","20170701235630Lh.jpeg","20170702133730Ch.jpeg","20170719235330Lh.jpeg","20170726235030Lh.jpeg","20170728150230Ch.jpeg","20170801164710Mh.jpeg","20170804140650Bh.jpeg","20170806133710Th.jpeg","20170809164910Mh.jpeg","20170827165310Mh.jpeg","20170831170750Bh.jpeg","20170925225630Lh.jpeg","20171005133730Ch.jpeg","20171005144050Bh.jpeg","20171007133710Th.jpeg","20171007133730Ch.jpeg","20171008144250Bh.jpeg","20171011170310Mh.jpeg","20171011224130Lh.jpeg","20171022170710Mh.jpeg","20171025133730Ch.jpeg","20171025223130Lh.jpeg","20171029150150Bh.jpeg","20171101133730Ch.jpeg","20171103222630Lh.jpeg","20171106171510Mh.jpeg","20171109023950Uh.jpeg","20171114133730Ch.jpeg","20171115170950Bh.jpeg","20171115222230Lh.jpeg","20171123133730Ch.jpeg","20171123222130Lh.jpeg","20180109223930Lh.jpeg","20180112130330Ch.jpeg","20180202130330Ch.jpeg","20180301050430Lh.jpeg","20180306081810Th.jpeg","20180318020350Uh.jpeg","20180318232230Lh.jpeg","20180506140230Ch.jpeg","20180710031950Uh.jpeg","20180712123630Ch.jpeg","20180714000010Mh.jpeg","20180804234530Lh.jpeg","20180805232510Mh.jpeg","20180827232530Lh.jpeg","20180923112330Ch.jpeg","20180924112030Ch.jpeg","20180926014350Uh.jpeg","20181014105830Ch.jpeg","20181023232430Lh.jpeg","20181024232430Lh.jpeg","20181101230010Mh.jpeg","20190317145150Bh.jpeg","20190322144450Bh.jpeg","20190412025430Lh.jpeg","20190602025430Lh.jpeg","20190611032810Mh.jpeg","20190629163710Mh.jpeg","20190704123830Ch.jpeg","20190805054730Lh.jpeg","20190925165810Mh.jpeg","20190926165810Mh.jpeg","20190927165910Mh.jpeg","20200105073250Uh.jpeg","20200114073230Lh.jpeg","20200116073250Uh.jpeg","20200806053830Lh.jpeg","20201022223330Lh.jpeg","20201227202710Mh.jpeg","20201228202430Ch.jpeg","20210222222850Bh.jpeg","20210223222830Ch.jpeg","20210329015250Uh.jpeg","20210329114430Ch.jpeg","20210331231510Mh.jpeg","20210620163530Mh.jpeg","20210621133650Bh.jpeg","20210624133950Bh.jpeg","20210626060050Uh.jpeg","20210626133950Bh.jpeg","20210721123430Ch.jpeg","20210728164530Mh.jpeg","20210731074710Th.jpeg","20210810121630Ch.jpeg","20210812044930Lh.jpeg","20210814044930Lh.jpeg","20210815140550Bh.jpeg","20211016145050Bh.jpeg","20211119103130Ch.jpeg","20211222120130Ch.jpeg","20211225025850Uh.jpeg","20211229120130Ch.jpeg","20211229120210Th.jpeg","20220104043050Uh.jpeg","20220122085210Th.jpeg","20220328143550Bh.jpeg","20220418020950Uh.jpeg","20220618185332Bh.jpeg","20220622205632Bh.jpeg","20220710085152Th.jpeg","20220718185352Mh.jpeg","20220721185332Bh.jpeg","20220724185352Mh.jpeg"]}''')
print({key: len(value) for key, value in SPLIT.items() if isinstance(value, list)})

In [ ]:
annotation_files = list(INPUT_ROOT.rglob('MAGFiLO_1.0_Annotations_kaggle2026_train.json'))
assert len(annotation_files) == 1, f'Attach the competition data; found {len(annotation_files)} annotation files'
DATA = annotation_files[0].parent.parent
assert (DATA / 'train/train_images').is_dir() and (DATA / 'test/test_images').is_dir(), sorted(p.name for p in DATA.iterdir())
print('data root:', DATA)
print('train images:', len(list((DATA / 'train/train_images').iterdir())), '| test images:', len(list((DATA / 'test/test_images').iterdir())))

## 1. Debug run (about 2 minutes)

In [ ]:
# Quick end-to-end check on a handful of images: confirms data paths, the model, tuning, scoring and CSV export.
debug_metrics = run_experiment(DATA, SPLIT, WORK_ROOT / 'debug', debug=True)
print('debug run finished: every stage ran. (PQ after 8 training steps on 4 images is meaningless, often 0.)')

## 2. Full GPU run (train, tune on calibration, score validation once, write candidate CSV)

In [ ]:
RUN_FULL = torch.cuda.is_available()   # set True to force a (very slow) CPU run
CFG = TrainConfig()                     # crop 512, batch 8, 40 epochs x 150 steps; edit here to experiment
if RUN_FULL:
    metrics = run_experiment(DATA, SPLIT, WORK_ROOT / 'run1', CFG, baseline_pq=0.1796)
else:
    print('Skipped: no GPU available.')

In [ ]:
if RUN_FULL:
    val = metrics['validation']
    print(f"held-out validation PQ: {val['pq']:.4f}  (95% CI {val['pq_bootstrap_95ci'][0]:.4f} to {val['pq_bootstrap_95ci'][1]:.4f})")
    print(f"classical baseline PQ:  {metrics['baseline_validation_pq']:.4f}")
    print(f"tuned post-processing:  {metrics['post']}")
    print(f"tp/fp/fn: {val['tp']} / {val['fp']} / {val['fn']}   images: {val['images']}   annotation sets: {val['annotation_sets']}")
    print('candidate CSV:', metrics['verification'])
    for record in metrics['history'][-5:]:
        print(record)